# WhisperSubs — Google Colab / Kaggle Notebook

Transcribe audio from YouTube, Twitch, or uploaded files using Whisper AI with multiple backend adapters.

## Quick Start
1. **Runtime** → **Change runtime type** → select **T4 GPU** (or better)  
2. Run each cell in order (Shift+Enter)
3. Paste URLs or upload files in the **Transcribe** section

## Supported Backends
| Prefix | Backend | Requires |
|--------|---------|----------|
| *(bare)* | faster-whisper (default) | — |
| `groq:` | Groq Cloud API | `GROQ_API_KEY` |
| `deepgram:` | Deepgram Cloud API | `DEEPGRAM_API_KEY` |
| `hf:` | Hugging Face pipeline | HF token for gated models |
| `whisperx:` | WhisperX (word-level + diarization) | — |
| `whispercpp:` | whisper.cpp Python bindings | — |
| `moonshine:` | Moonshine (ONNX) | — |
| `parakeet:` | NVIDIA NeMo Parakeet | HF token |
| `canary:` | NVIDIA NeMo Canary | HF token |
| `chirp:` | Google Chirp (Vertex AI) | GCP credentials |
| `voxtral:` | Mistral Voxtral | `VOXTRAL_API_KEY` |
| `vibevoice:` | VibeVoice | `VIBEVOICE_API_KEY` |
| `whisperturbo:` | WhisperTurbo | — |

In [ ]:
#@title 1. Install Dependencies { display-mode: "form" }
#@markdown Runs automatically. Takes ~3 min on first run.

import os, sys, subprocess

def _run(cmd):
    return subprocess.run(cmd, shell=True, capture_output=True, text=True)

# Detect environment
IS_COLAB = 'google.colab' in str(get_ipython()) if hasattr(get_ipython(), '__module__') else False
IS_KAGGLE = os.path.exists('/kaggle')
print(f"Environment: {'Colab' if IS_COLAB else 'Kaggle' if IS_KAGGLE else 'Local'}")

# Install ffmpeg (required for audio extraction)
if IS_COLAB:
    _run('apt-get update -qq && apt-get install -y -qq ffmpeg')
elif IS_KAGGLE:
    _run('apt-get update -qq && apt-get install -y -qq ffmpeg')
else:
    if not _run('which ffmpeg').stdout.strip():
        print('WARNING: ffmpeg not found — install it manually')

# PyTorch with CUDA (Colab/Kaggle usually have it pre-installed, but ensure)
if IS_COLAB or IS_KAGGLE:
    try:
        import torch
        if not torch.cuda.is_available():
            print('WARNING: GPU not detected. Go to Runtime > Change runtime type > T4 GPU')
        else:
            print(f'GPU: {torch.cuda.get_device_name(0)}')
    except ImportError:
        _run('pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu121')

# Core dependencies
CORE_DEPS = [
    'faster-whisper',
    'yt-dlp',
    'pymkv',
    'requests',
    'numpy',
    'natsort',
    'tqdm',
    'psutil',
    'pyperclip',
]

print('Installing core dependencies...')
for dep in CORE_DEPS:
    result = _run(f'pip install --quiet {dep}')
    if result.returncode != 0:
        print(f'  FAILED: {dep}\n    {result.stderr.strip()[:200]}')
    else:
        print(f'  OK: {dep}')

print('Core dependencies installed.')

In [ ]:
#@title 2. Install Adapter Backends (optional) { display-mode: "form" }
#@markdown Check the adapters you want to install. Unchecked ones are skipped.

install_whisperx = False #@param {type:"boolean"}
install_moonshine = False #@param {type:"boolean"}
install_whispercpp = False #@param {type:"boolean"}
install_nemo = False #@param {type:"boolean"}
install_groq_sdk = False #@param {type:"boolean"}
install_deepgram_sdk = False #@param {type:"boolean"}

ADAPTER_DEPS = {
    'whisperx': [('whisperx', 'whisperx')],
    'moonshine': [('moonshine', 'moonshine')],
    'whispercpp': [('whisper-cpp-python', 'whispercpp')],
    'nemo': [('nemo_toolkit[asr]', 'nemo')],
    'groq_sdk': [('groq', 'groq')],
    'deepgram_sdk': [('deepgram-sdk', 'deepgram')],
}

flags = {
    'whisperx': install_whisperx,
    'moonshine': install_moonshine,
    'whispercpp': install_whispercpp,
    'nemo': install_nemo,
    'groq_sdk': install_groq_sdk,
    'deepgram_sdk': install_deepgram_sdk,
}

for key, enabled in flags.items():
    if not enabled:
        continue
    for pip_name, label in ADAPTER_DEPS[key]:
        print(f'Installing {label}...')
        result = __import__('subprocess').run(
            f'pip install --quiet {pip_name}',
            shell=True, capture_output=True, text=True
        )
        if result.returncode != 0:
            print(f'  FAILED: {label}\n    {result.stderr.strip()[:300]}')
        else:
            print(f'  OK: {label}')

print('Adapter installation done.')

In [ ]:
#@title 3. Clone WhisperSubs { display-mode: "form" }
#@markdown The repo is cloned (or updated) and added to `sys.path` so all modules are importable.

import os, sys, subprocess

REPO_URL = 'https://github.com/Hslffd/.whisper-subs.git'
REPO_DIR = '/content/whisper-subs' if os.path.exists('/content') else os.path.join(os.getcwd(), 'whisper-subs')

if os.path.isdir(os.path.join(REPO_DIR, '.git')):
    print(f'Repo exists at {REPO_DIR}, pulling latest...')
    subprocess.run(['git', '-C', REPO_DIR, 'pull', '--ff-only'], capture_output=True)
else:
    print(f'Cloning repo to {REPO_DIR}...')
    subprocess.run(['git', 'clone', '--depth', '1', REPO_URL, REPO_DIR], check=True)

if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)

os.chdir(REPO_DIR)
print(f'Working directory: {os.getcwd()}')
print('Ready to import whisper_subs.')

In [ ]:
#@title 4. API Keys (optional) { display-mode: "form" }
#@markdown Only fill in keys for cloud/API backends you plan to use.
#@markdown Keys are set as environment variables for this session only.

import os

GROQ_API_KEY = '' #@param {type:"string"}
DEEPGRAM_API_KEY = '' #@param {type:"string"}
HF_TOKEN = '' #@param {type:"string"}
GOOGLE_APPLICATION_CREDENTIALS = '' #@param {type:"string"}

if GROQ_API_KEY:
    os.environ['GROQ_API_KEY'] = GROQ_API_KEY
if DEEPGRAM_API_KEY:
    os.environ['DEEPGRAM_API_KEY'] = DEEPGRAM_API_KEY
if HF_TOKEN:
    os.environ['HF_TOKEN'] = HF_TOKEN
    os.environ['HUGGING_FACE_HUB_TOKEN'] = HF_TOKEN
if GOOGLE_APPLICATION_CREDENTIALS:
    os.environ['GOOGLE_APPLICATION_CREDENTIALS'] = GOOGLE_APPLICATION_CREDENTIALS

set_keys = [k for k in ['GROQ_API_KEY', 'DEEPGRAM_API_KEY', 'HF_TOKEN', 'GOOGLE_APPLICATION_CREDENTIALS'] if os.environ.get(k)]
print(f'API keys set: {set_keys}' if set_keys else 'No API keys configured (local models only).')

In [ ]:
#@title 5. List Available Models { display-mode: "form" }
#@markdown Shows all model names/indices and which adapters are actually available.

from model import list_available_models, TranscriptionContext

print('=' * 60)
list_available_models()

print('\n' + '=' * 60)
print('Adapter availability check:')
ctx = TranscriptionContext()
for a in ctx.list_available_adapters():
    status = 'AVAILABLE' if a['available'] else 'NOT AVAILABLE'
    print(f'  [{status}] {a["prefix"]}: {a["name"]} ({len(a["models"])} models)')

---
## Transcribe

Use one of the three methods below: URL, upload, or Kaggle dataset path.

In [ ]:
#@title 6a. Transcribe from URL { display-mode: "form" }
#@markdown Paste a YouTube/Twitch URL or any audio/video URL.
#@markdown Separate multiple URLs with newlines.

URL = '' #@param {type:"string"}
MODEL = 'large-v3' #@param ['tiny', 'base', 'small', 'medium', 'large', 'large-v2', 'large-v3', 'distil-whisper/distil-large-v3', 'distil-whisper/distil-large-v2', 'distil-whisper/distil-medium.en', 'distil-whisper/distil-small.en', 'groq:whisper-large-v3', 'groq:whisper-large-v3-turbo', 'deepgram:nova-3', 'deepgram:whisper-turbo', 'whisperx:large-v3', 'hf:openai/whisper-large-v3']
DEVICE = 'cuda' #@param ['cuda', 'cpu']
COMPUTE_TYPE = 'float16' #@param ['int8', 'float16', 'int8_float16']
LANGUAGE = '' #@param {type:"string"}
FORCE = False #@param {type:"boolean"}
VAD = True #@param {type:"boolean"}
DIARIZATION = False #@param {type:"boolean"}
MERGE_LINES = False #@param {type:"boolean"}

from whisper_subs import WhisperSubs

if not URL.strip():
    raise ValueError('Please enter a URL above.')

urls = [u.strip() for u in URL.strip().split('\n') if u.strip()]

processor = WhisperSubs(
    model_name=MODEL,
    device=DEVICE,
    compute_type=COMPUTE_TYPE,
    force=FORCE,
    sub_lang=LANGUAGE or None,
    vad_filter=VAD,
    diarization=DIARIZATION,
    merge_lines=MERGE_LINES,
)

for url in urls:
    print(f'\n--- Processing: {url} ---')
    processor.process(url)

print(f'\nDone. Output saved to ~/Documents/Youtube-Subs/')

In [ ]:
#@title 6b. Transcribe Uploaded File { display-mode: "form" }
#@markdown Upload an audio/video file from your local machine.

MODEL = 'large-v3' #@param ['tiny', 'base', 'small', 'medium', 'large', 'large-v2', 'large-v3', 'distil-whisper/distil-large-v3', 'distil-whisper/distil-large-v2', 'distil-whisper/distil-medium.en', 'distil-whisper/distil-small.en', 'groq:whisper-large-v3', 'groq:whisper-large-v3-turbo', 'deepgram:nova-3', 'deepgram:whisper-turbo', 'whisperx:large-v3', 'hf:openai/whisper-large-v3']
DEVICE = 'cuda' #@param ['cuda', 'cpu']
COMPUTE_TYPE = 'float16' #@param ['int8', 'float16', 'int8_float16']
LANGUAGE = '' #@param {type:"string"}
FORCE = False #@param {type:"boolean"}
VAD = True #@param {type:"boolean"}
DIARIZATION = False #@param {type:"boolean"}
MERGE_LINES = False #@param {type:"boolean"}

from whisper_subs import WhisperSubs

try:
    from google.colab import files
    uploaded = files.upload()
    file_paths = list(uploaded.keys())
except ImportError:
    # Kaggle or local: use a file dialog or prompt
    import glob
    data_dir = '/kaggle/input' if os.path.exists('/kaggle/input') else os.path.expanduser('~/Documents')
    candidates = sorted(glob.glob(os.path.join(data_dir, '**', '*.*'), recursive=True))
    audio_exts = {'.mp3', '.wav', '.flac', '.m4a', '.ogg', '.opus', '.mp4', '.mkv', '.webm', '.avi'}
    file_paths = [f for f in candidates if os.path.splitext(f)[1].lower() in audio_exts]
    if not file_paths:
        raise FileNotFoundError(f'No audio/video files found in {data_dir}')
    print(f'Found {len(file_paths)} file(s):')
    for f in file_paths:
        print(f'  {f}')

processor = WhisperSubs(
    model_name=MODEL,
    device=DEVICE,
    compute_type=COMPUTE_TYPE,
    force=FORCE,
    sub_lang=LANGUAGE or None,
    vad_filter=VAD,
    diarization=DIARIZATION,
    merge_lines=MERGE_LINES,
)

for fp in file_paths:
    print(f'\n--- Processing: {fp} ---')
    processor.process(fp)

print(f'\nDone. Output saved to ~/Documents/Youtube-Subs/')

In [ ]:
#@title 6c. Transcribe from Kaggle Dataset { display-mode: "form" }
#@markdown Enter the path to a Kaggle dataset directory (e.g. `/kaggle/input/my-dataset/`).
#@markdown All audio/video files in that directory will be transcribed.

DATASET_PATH = '/kaggle/input/' #@param {type:"string"}
MODEL = 'large-v3' #@param ['tiny', 'base', 'small', 'medium', 'large', 'large-v2', 'large-v3', 'distil-whisper/distil-large-v3', 'distil-whisper/distil-large-v2', 'distil-whisper/distil-medium.en', 'distil-whisper/distil-small.en', 'groq:whisper-large-v3', 'groq:whisper-large-v3-turbo', 'deepgram:nova-3', 'deepgram:whisper-turbo', 'whisperx:large-v3', 'hf:openai/whisper-large-v3']
DEVICE = 'cuda' #@param ['cuda', 'cpu']
COMPUTE_TYPE = 'float16' #@param ['int8', 'float16', 'int8_float16']
LANGUAGE = '' #@param {type:"string"}
FORCE = False #@param {type:"boolean"}
VAD = True #@param {type:"boolean"}
DIARIZATION = False #@param {type:"boolean"}
MERGE_LINES = False #@param {type:"boolean"}

import os, glob
from whisper_subs import WhisperSubs

if not os.path.isdir(DATASET_PATH):
    raise FileNotFoundError(f'Directory not found: {DATASET_PATH}')

audio_exts = {'.mp3', '.wav', '.flac', '.m4a', '.ogg', '.opus', '.mp4', '.mkv', '.webm', '.avi', '.wma', '.aac'}
file_paths = sorted(
    f for f in glob.glob(os.path.join(DATASET_PATH, '**', '*.*'), recursive=True)
    if os.path.splitext(f)[1].lower() in audio_exts
)

if not file_paths:
    raise FileNotFoundError(f'No audio/video files found in {DATASET_PATH}')

print(f'Found {len(file_paths)} file(s) in {DATASET_PATH}:')
for f in file_paths:
    print(f'  {f}')

processor = WhisperSubs(
    model_name=MODEL,
    device=DEVICE,
    compute_type=COMPUTE_TYPE,
    force=FORCE,
    sub_lang=LANGUAGE or None,
    vad_filter=VAD,
    diarization=DIARIZATION,
    merge_lines=MERGE_LINES,
)

for fp in file_paths:
    print(f'\n--- Processing: {fp} ---')
    processor.process(fp)

print(f'\nDone. Output saved to ~/Documents/Youtube-Subs/')

---
## Results

View and download the generated subtitle files.

In [ ]:
#@title 7. View & Download Results { display-mode: "form" }
#@markdown Lists all generated .srt and helper files, then downloads them.

import os, glob, pathlib

OUTPUT_DIR = os.path.join(os.path.expanduser('~'), 'Documents', 'Youtube-Subs')

srt_files = sorted(glob.glob(os.path.join(OUTPUT_DIR, '**', '*.srt'), recursive=True))
helper_files = sorted(glob.glob(os.path.join(OUTPUT_DIR, '**', '*.htm'), recursive=True)
    + glob.glob(os.path.join(OUTPUT_DIR, '**', '*.sh'), recursive=True)
    + glob.glob(os.path.join(OUTPUT_DIR, '**', '*.bat'), recursive=True)
    + glob.glob(os.path.join(OUTPUT_DIR, '**', '*.ps1'), recursive=True)
    + glob.glob(os.path.join(OUTPUT_DIR, '**', '*.m3u'), recursive=True))

print(f'=== Subtitle files ({len(srt_files)}) ===')
for f in srt_files:
    size_kb = os.path.getsize(f) / 1024
    print(f'  {f}  ({size_kb:.1f} KB)')

if helper_files:
    print(f'\n=== Helper files ({len(helper_files)}) ===')
    for f in helper_files:
        print(f'  {f}')

all_files = srt_files + helper_files

# Download in Colab
try:
    from google.colab import files as colab_files
    if all_files:
        print(f'\nDownloading {len(all_files)} file(s)...')
        for f in all_files:
            colab_files.download(f)
except ImportError:
    print(f'\nFiles are in: {OUTPUT_DIR}')
    print('Copy them from the Kaggle output or working directory.')

In [ ]:
#@title 8. CLI Mode (advanced) { display-mode: "form" }
#@markdown Direct CLI access to whisper_subs.py with full argument support.
#@markdown Edit the command string below. All `whisper_subs.py` flags work.

CLI_COMMAND = 'large-v3 https://www.youtube.com/watch?v=dQw4w9WgXcQ' #@param {type:"string"}

import subprocess, sys, os

REPO_DIR = '/content/whisper-subs' if os.path.exists('/content/whisper-subs') else os.path.join(os.getcwd(), 'whisper-subs')
script = os.path.join(REPO_DIR, 'whisper_subs.py')

cmd = [sys.executable, script] + CLI_COMMAND.strip().split()
print(f'Running: {" ".join(cmd)}')
print('=' * 60)
result = subprocess.run(cmd, cwd=REPO_DIR)
print('=' * 60)
print(f'Exit code: {result.returncode}')

---
## Batch Processing

Process multiple URLs from a text list (one per line).

In [ ]:
#@title 9. Batch Process URL List { display-mode: "form" }
#@markdown Paste URLs (one per line) or upload a .txt file with URLs.
#@markdown Lines starting with `#` are ignored. Append a model name after a URL: `URL model_name`

URL_LIST = '''\
# Paste URLs below, one per line:
# https://www.youtube.com/watch?v=example1
# https://www.youtube.com/watch?v=example2 large-v3
''' #@param {type:"string"}

DEFAULT_MODEL = 'large-v3' #@param ['tiny', 'base', 'small', 'medium', 'large', 'large-v2', 'large-v3', 'distil-whisper/distil-large-v3', 'groq:whisper-large-v3', 'deepgram:nova-3']
DEVICE = 'cuda' #@param ['cuda', 'cpu']
COMPUTE_TYPE = 'float16' #@param ['int8', 'float16']
SHUFFLE = False #@param {type:"boolean"}

import os, random, datetime
from whisper_subs import WhisperSubs, read_sources_from_file

# Write URL list to a temp process file
OUTPUT_DIR = os.path.join(os.path.expanduser('~'), 'Documents', 'Youtube-Subs')
os.makedirs(OUTPUT_DIR, exist_ok=True)

ts = datetime.datetime.now().strftime('%Y%m%d_%H%M%S')
process_file = os.path.join(OUTPUT_DIR, f'batch_{ts}.txt')
with open(process_file, 'w') as f:
    f.write(URL_LIST)

# Reuse the process-file logic from whisper_subs CLI
sources = read_sources_from_file(process_file)
if not sources:
    raise ValueError('No valid sources found in the URL list.')

if SHUFFLE:
    random.shuffle(sources)
    print(f'Shuffled {len(sources)} sources')

print(f'Processing {len(sources)} source(s)...')

for i, source_info in enumerate(sources):
    model = source_info.get('model') or DEFAULT_MODEL
    from model import getName
    model_name = getName(model)
    processor = WhisperSubs(
        model_name=model_name,
        device=DEVICE,
        compute_type=COMPUTE_TYPE,
    )
    print(f'\n[{i+1}/{len(sources)}] {source_info["url"]} (model: {model_name})')
    processor.process(source_info['url'])

print(f'\nBatch complete. Process file saved to: {process_file}')

## Tips

- **GPU required** for local models. Use `Runtime > Change runtime type > T4 GPU` in Colab, or enable GPU in Kaggle notebook settings.
- **`float16`** compute is recommended for GPU; **`int8`** for CPU.
- **Distil-Whisper** models (`distil-whisper/distil-large-v3`) are ~6x faster with minimal quality loss.
- **Cloud APIs** (Groq, Deepgram) are faster but require API keys and have usage costs.
- **Diarization** (`--diarization` / DIARIZATION=True) identifies speakers but requires more memory.
- Output is saved to `~/Documents/Youtube-Subs/` organized by channel name.
- Re-run cell 7 to download results after transcription.